In [ ]:
# --- Cell 1: imports
import os, time, json, datetime as dt
from typing import List, Dict, Any, Iterable

import requests
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, min as spark_min, max as spark_max
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType

# prettier plots
plt.rcParams["figure.dpi"] = 110


In [2]:
# --- Cell 2: edit these as needed
# Elhub API base + endpoint path:
# Check your portal; if your dataset path differs, adjust ENDPOINT_PATH.
ELHUB_API_BASE = "https://api.elhub.no/energy-data-api"
ENDPOINT_PATH   = "PRODUCTION_PER_GROUP_MBA_HOUR"  # change if your API uses another path segment

# Auth (set env vars or paste the token/key directly for quick testing)
ELHUB_API_TOKEN = os.environ.get("ELHUB_API_TOKEN")  # "Bearer-Style-TokenString"
ELHUB_API_KEY   = os.environ.get("ELHUB_API_KEY")    # subscription key if used

headers = {"Accept": "application/json"}
if ELHUB_API_TOKEN:
    headers["Authorization"] = f"Bearer {ELHUB_API_TOKEN}"
if ELHUB_API_KEY:
    headers["Ocp-Apim-Subscription-Key"] = ELHUB_API_KEY

DATASET_NAME = "PRODUCTION_PER_GROUP_MBA_HOUR"
YEAR = 2021

# Time range in UTC
year_start_utc = dt.datetime(YEAR, 1, 1, 0, 0, 0, tzinfo=dt.timezone.utc)
year_end_utc   = dt.datetime(YEAR+1, 1, 1, 0, 0, 0, tzinfo=dt.timezone.utc)  # exclusive

# Dataset may have a max period per request; start conservatively (7 days). Increase once verified.
MAX_WINDOW_DAYS = 7

# Spark connector package coordinates (Spark 3.5 / Scala 2.12)
MONGO_PKG = "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0"
CASSANDRA_PKG = "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0"

# Cassandra connection (edit if not local)
CASSANDRA_HOST = os.environ.get("CASSANDRA_HOST", "127.0.0.1")
CASSANDRA_PORT = os.environ.get("CASSANDRA_PORT", "9042")

# MongoDB URI (local or Atlas). Example local: "mongodb://localhost:27017"
MONGO_URI = os.environ.get("MONGO_URI", "mongodb://localhost:27017")
MONGO_DB  = "elhub"
MONGO_COLL = "production_2021"

# Choose plot area
PLOT_PRICE_AREA = "NO1"  # change to NO2/NO3/NO4/NO5 as you like


In [3]:
# --- Cell 3: Spark session with connectors via --packages

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")  # add this on Windows
    .appName("elhub-2021-production")
    .config("spark.sql.session.timeZone", "UTC")
    # Use local JARs to avoid Maven/Hadoop/winutils on Windows
    .config(
        "spark.jars",
        r"C:\\spark-jars\\mongo-spark-connector_2.12-10.3.0.jar,C:\\spark-jars\\spark-cassandra-connector-assembly_2.12-3.5.0.jar"
    )
    # Cassandra connection
    .config("spark.cassandra.connection.host", CASSANDRA_HOST)
    .config("spark.cassandra.connection.port", CASSANDRA_PORT)
    .getOrCreate()
)

spark


Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://wiki.apache.org/hadoop/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:735)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:270)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1139)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1125)
	at org.apache.spark.util.Utils$.fetchFile(Utils.scala:489)
	at org.apache.spark.executor.Executor.$anonfun$updateDependencies$14(Executor.scala:1163)
	at org.apache.spark.executor.Executor.$anonfun$updateDependencies$14$adapted(Executor.scala:1155)
	at scala.collection.TraversableLike$WithFilter.$anonfun$foreach$1(TraversableLike.scala:985)
	at scala.collection.immutable.Map$Map2.foreach(Map.scala:273)
	at scala.collection.TraversableLike$WithF

In [4]:
# --- New Cell: quick JAR path pre-check
import os
jar_paths = [
    r"C:\\spark-jars\\mongo-spark-connector_2.12-10.3.0.jar",
    r"C:\\spark-jars\\spark-cassandra-connector-assembly_2.12-3.5.0.jar",
]

for p in jar_paths:
    if os.path.exists(p):
        size = os.path.getsize(p)
        print(f"FOUND: {p} — {size:,} bytes")
    else:
        print(f"MISSING: {p} — adjust path or download compatible JAR")

print("\nIf found, restart the kernel and re-run the Spark session cell to pick up the jars.")


FOUND: C:\\spark-jars\\mongo-spark-connector_2.12-10.3.0.jar — 191,973 bytes
FOUND: C:\\spark-jars\\spark-cassandra-connector-assembly_2.12-3.5.0.jar — 15,219,525 bytes

If found, restart the kernel and re-run the Spark session cell to pick up the jars.


In [ ]:
# --- Cell 4: helpers
def utc_iso(ts: dt.datetime) -> str:
    return ts.astimezone(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def windows(start_utc: dt.datetime, end_utc: dt.datetime, days: int):
    cur = start_utc
    step = dt.timedelta(days=days)
    while cur < end_utc:
        nxt = min(cur + step, end_utc)
        yield cur, nxt
        cur = nxt

def request_with_retry(url: str, params: Dict[str, Any], headers: Dict[str, str], retries=3, backoff=1.6):
    for i in range(retries):
        r = requests.get(url, params=params, headers=headers, timeout=60)
        if r.status_code == 200:
            return r.json()
        if i < retries - 1:
            time.sleep(backoff ** (i+1))
    r.raise_for_status()

def parse_records(payload: Dict[str, Any]) -> List[Dict[str, Any]]:
    # The assignment: “Extract only the list in productionPerGroupMbaHour”
    return payload.get("productionPerGroupMbaHour", []) or []


In [ ]:
# --- Cell 5: download
endpoint = f"{ELHUB_API_BASE}/{ENDPOINT_PATH}"
all_rows: List[Dict[str, Any]] = []

for t0, t1 in windows(year_start_utc, year_end_utc, MAX_WINDOW_DAYS):
    params = {
        "timeFrom": utc_iso(t0),
        "timeTo":   utc_iso(t1),
        # If your API requires explicit areas, uncomment:
        # "priceAreas": "NO1,NO2,NO3,NO4,NO5"
    }
    payload = request_with_retry(endpoint, params=params, headers=headers)
    rows = parse_records(payload)
    all_rows.extend(rows)

len(all_rows)


In [ ]:
# --- Cell 6: pandas normalization
if not all_rows:
    raise RuntimeError("No data returned. Check endpoint/auth/params and time-window length.")

df = pd.json_normalize(all_rows)

required = ["priceArea", "productionGroup", "startTime", "quantityKwh"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns from API response: {missing}")

df = df[required].copy()
df["startTime"]   = pd.to_datetime(df["startTime"], utc=True, errors="coerce")
df["quantityKwh"] = pd.to_numeric(df["quantityKwh"], errors="coerce")

display(df.head())
print(df.dtypes)
print("rows:", len(df), "| start:", df["startTime"].min(), "| end:", df["startTime"].max())


In [ ]:
# --- Cell 7: Spark DataFrame with schema
schema = StructType([
    StructField("priceArea",       StringType(),    False),
    StructField("productionGroup", StringType(),    False),
    StructField("startTime",       TimestampType(), False),
    StructField("quantityKwh",     DoubleType(),    True),
])

sdf = spark.createDataFrame(df, schema=schema).dropna(subset=["priceArea","productionGroup","startTime"])
sdf.cache()
sdf.count(), sdf.select(spark_min("startTime"), spark_max("startTime")).first()


In [ ]:
# --- Cell 8: Spark → Cassandra
(
    sdf.write
    .format("org.apache.spark.sql.cassandra")
    .options(keyspace="elhub", table="production_hourly")
    .mode("append")
    .save()
)
print("✅ Wrote to Cassandra: elhub.production_hourly")


In [ ]:
# --- Cell 9: Read back four columns
sdf_c = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(keyspace="elhub", table="production_hourly")
    .load()
    .select("priceArea", "productionGroup", "startTime", "quantityKwh")
)

sdf_c.printSchema()
sdf_c.show(5, truncate=False)


In [ ]:
# --- Cell 10: prep pandas for plotting
pdf = sdf_c.filter(col("priceArea") == PLOT_PRICE_AREA).toPandas()
pdf["startTime"] = pd.to_datetime(pdf["startTime"], utc=True)

pdf_year = pdf[(pdf["startTime"] >= "2021-01-01") & (pdf["startTime"] < "2022-01-01")].copy()

# PIE: total by productionGroup (full year)
pie_data = (
    pdf_year.groupby("productionGroup", as_index=False)["quantityKwh"]
    .sum()
    .sort_values("quantityKwh", ascending=False)
)

plt.figure(figsize=(6,6))
plt.pie(
    pie_data["quantityKwh"].fillna(0.0),
    labels=pie_data["productionGroup"],
    autopct="%1.1f%%",
    startangle=90
)
plt.title(f"Total production in 2021 by group — {PLOT_PRICE_AREA}")
plt.tight_layout()
plt.show()


In [ ]:
# --- Cell 11: LINE: Jan 2021, one line per productionGroup
pdf_jan = pdf_year[pdf_year["startTime"].dt.month == 1].copy()

line_df = (
    pdf_jan
    .pivot_table(index="startTime", columns="productionGroup", values="quantityKwh", aggfunc="sum")
    .sort_index()
)

plt.figure(figsize=(12,4))
for colname in line_df.columns:
    plt.plot(line_df.index, line_df[colname], label=colname)
plt.title(f"Hourly production — {PLOT_PRICE_AREA} — Jan 2021 (UTC)")
plt.xlabel("Time (UTC)")
plt.ylabel("kWh")
plt.legend(ncol=4, fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# --- Cell 12A: Spark → MongoDB (uses mongo-spark-connector)
(
    sdf_c.write
    .format("mongodb")
    .mode("overwrite")  # or "append"
    .option("uri", f"{MONGO_URI}/{MONGO_DB}.{MONGO_COLL}")
    .save()
)
print(f"✅ Wrote to MongoDB: {MONGO_DB}.{MONGO_COLL}")


In [ ]:
# --- Cell 13: sanity checks
sdf_c.groupBy("priceArea").agg(
    spark_min("startTime").alias("minTime"),
    spark_max("startTime").alias("maxTime"),
).show(truncate=False)

sdf_c.filter(col("priceArea")==PLOT_PRICE_AREA) \
     .groupBy("productionGroup") \
     .agg(spark_sum("quantityKwh").alias("totalKwh")) \
     .orderBy(col("totalKwh").desc()) \
     .show(truncate=False)
